# पाठ 11 - एजंट-टू-एजंट (A2A) प्रोटोकॉल


## सेटअप


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## A2A प्रोटोकॉल म्हणजे काय?

The **Agent-to-Agent (A2A) protocol** हा एक खुला मानक आहे जो AI एजंटांना संवाद साधण्यास, एकमेकांना शोधण्यास आणि सहकार्य करण्यास सक्षम करतो — अगदी जे वेगळ्या फ्रेमवर्कवर तयार केलेले किंवा वेगळ्या सेवांवर होस्ट केलेले असले तरीही.

Key concepts:

- **Discovery** – एजंट त्यांच्या क्षमता वर्णन करणारे *Agent Card* प्रकाशित करतात, ज्यामुळे इतर एजंटांना (किंवा समन्वयकांना) एखाद्या कामासाठी योग्य विशेषज्ञ सापडणे सोपे होते.
- **Message Passing** – एजंट सामान्य प्रोटोकॉलद्वारे संरचित संदेशांची देवाणघेवाण करतात, त्यामुळे एका एजंटकडून आलेली विनंती दुसऱ्या एजंटद्वारे त्याच्या अंतर्गत अंमलबजावणीच्या भिन्नतेकडे दुर्लक्ष करत समजून घेऊन पूर्ण केली जाऊ शकते.
- **Task Lifecycle** – A2A ही *सादर*, *काम चालू*, *पूर्ण*, आणि *अयशस्वी* अशा अवस्थांची व्याख्या करते, ज्यामुळे समन्वयकाला कोणत्या प्रकारे नेमलेल्या कार्याची प्रगती होत आहे याची पूर्ण दृश्यता मिळते.

In this lesson we simulate A2A-style collaboration by wiring three specialized travel agents into a workflow where each agent contributes its expertise and passes results to the next.


## विशेषीकृत प्रवास एजंट तयार करणे


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## वर्कफ्लोद्वारे बहु-एजंट सहकार्य

आम्ही तीन एजंट्सना अनुक्रमिक वर्कफ्लोमध्ये जोडतो जे A2A मेसेज पासिंगचे प्रतिबिंब आहे:

1. **CurrencyExchangeAgent** वापरकर्त्याची विनंती प्राप्त करते आणि चलनाबद्दल मार्गदर्शन प्रदान करते.
2. **ActivityPlannerAgent** समृद्ध संदर्भ प्राप्त करते आणि क्रियाकलाप शिफारसी जोडते.
3. **TravelManagerAgent** दोन्ही इनपुटचे समाकलन करून अंतिम प्रवास सारांश तयार करते.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## उत्पादनात A2A समजून घेणे

उत्पादन वातावरणात A2A प्रोटोकॉल शक्तिशाली क्रॉस-सेवा परिस्थिती सक्षम करतो:

| क्षमता | वर्णन |
|---|---|
| **फ्रेमवर्क-आंतरपरस्परसंवाद** | एका फ्रेमवर्कने बनवलेला एजंट कोणत्याही दुसऱ्या A2A-सुसंगत फ्रेमवर्कने बनवलेल्या एजंटला काम सोपवू शकतो, ज्यामुळे खऱ्या अर्थाने संघटनांदरम्यान परस्परसंवाद शक्य होतो. |
| **सेवा सीमा** | एजंट वेगळ्या मायक्रोसर्व्हिसेस, क्लाउड प्रदेश किंवा अगदी भिन्न संस्थांमध्ये असू शकतात आणि तरीही सुरळीतपणे सहकार्य करू शकतात. |
| **डायनॅमिक शोध** | एक ऑर्केस्ट्रेटर रनटाइममध्ये Agent Card रजिस्ट्रीला चौकशी करू शकतो जेणेकरून दिलेल्या उपकार्यासाठी सर्वात योग्य विशेषज्ञ सापडू शकेल. |
| **स्ट्रीमिंग आणि पुश सूचना** | A2A रिअल-टाइम प्रगती अद्यतनांसाठी Server-Sent Events (SSE) आणि दीर्घकालीन कार्यांसाठी पुश सूचना समर्थित करते. |

वरील आम्ही तयार केलेला कार्यप्रवाह हा या नमुन्याचा एक सरलीकृत, इन-प्रोसेस आवृत्ती आहे. वास्तविक परिनियोजनात प्रत्येक एजंट एक HTTP endpoint उघडेल, Agent Card प्रकाशित करेल, आणि A2A JSON-RPC प्रोटोकॉलद्वारे संवाद साधेल.


## सारांश

या धड्यात आपण शिकलात:

1. **A2A प्रोटोकॉल काय आहे** — एजंट-ते-एजंट शोध, संदेशवहन,
   आणि कार्य व्यवस्थापन.
2. **विशेषीकृत एजंट कसे तयार करायचे** — एक चलन विनिमय एजंट, एक अ‍ॅक्टिव्हिटी प्लॅनर एजंट,
   आणि एक प्रवास व्यवस्थापक ऑर्केस्ट्रेटर.
3. **एजंटना वर्कफ्लोमध्ये कसे जोडायचे** — `WorkflowBuilder` वापरून क्रमिक
   एजंट्सदरम्यान संदेश देवाणघेवाणाचे मॉडेल तयार करणे.
4. **A2A उत्पादनात कसे कार्य करते** — क्रॉस-फ्रेमवर्क, क्रॉस-सर्व्हिस सहयोग सक्षम करणे
   डायनॅमिक शोध आणि स्ट्रीमिंग अद्यतनांसह.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
अस्वीकरण:
हा दस्तऐवज एआय भाषांतर सेवा Co-op Translator (https://github.com/Azure/co-op-translator) वापरून भाषांतरित केला गेलाय. आम्ही अचूकतेसाठी प्रयत्न करतो, तरी कृपया लक्षात ठेवा की स्वयंचलित भाषांतरांमध्ये चुका किंवा अचूकतेच्या त्रुटी असू शकतात. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्रोत मानला जावा. महत्त्वाच्या माहितीसाठी व्यावसायिक मानवी भाषांतर शिफारसीय आहे. या भाषांतराच्या वापरामुळे उद्भवणाऱ्या कोणत्याही गैरसमजुती किंवा चुकीच्या अर्थनिर्वचनांसाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
